# 実験音韻論の方法 ── 音韻理論をデータで検証する

### 第4章 音響分析の基礎 ── 分析コード集


ここに掲載されているコードは、
『実験音韻論の方法 ── 音韻理論をデータで検証する』(音声学ライブラリ 第4巻)の
分析コード集の一部である。MIT Licenseで公開しており、自由に複製・改変・
再配布してよい(詳細はサポートリポジトリのLICENSEを参照)。

このファイルを開いた時点では閲覧用として開くが、**ファイル → ドライブに
コピーを保存**を行えば、あなた自身のGoogle Driveに保存されたあなただけの
ファイルになる。自由に変更・実行・保存してよい。

**本文で報告されている数値・図は、すべてこのノートブックのコードを実際に
実行して得られたものである。** 架空データ(シミュレーション)を使っている
箇所は、各コードセルの直前のセルにその旨を明記してある。実データを模した
説明にすぎない部分と、実際の統計的性質を確認するための実行例とを、
本書は厳格に区別しており、このノートブックもその区別をそのまま保持している。

実行前に、Colabメニューの「ランタイム」→「ランタイムのタイプを変更」で
言語を **R** に切り替えること。既定のPythonランタイムのままだと
以降のセルがすべてエラーになる。


## 共通セットアップ

以降の各セルは、本書の`scripts/R/`に収録されている実際のコードと一字一句同じである(差し替えたのはこのセットアップだけ)。まずこのセルを実行してから、上から順に進めること。

In [ ]:
# ---- 図の日本語文字化け対策 ----
# Colabの既定Rランタイムのフォントには日本語グリフが無く、図中の日本語
# (タイトル・軸ラベルなど)が文字化け(豆腐記号)して表示される。showtextで
# Google Fontsの日本語フォントを取得し、"notosans"という名前で登録することで
# 回避する("sans"などの予約名はshowtextから上書きできないため専用名で登録する)。
if (!requireNamespace("showtext", quietly = TRUE)) install.packages("showtext", repos = "https://cloud.r-project.org")
library(showtext)
font_add_google("Noto Sans JP", "notosans")
showtext_auto()
showtext_opts(dpi = 100)

# ---- 共通セットアップ(scripts/R/00_setup.RのColab版) ----
# 本書のすべての解析スクリプトが冒頭で読み込む初期化処理を、Colab向けに
# 移植したもの。以降の章のコードは、本書のscripts/R/chNN_*.Rと一字一句同じ
# ものを、このセットアップの後に続けて実行する形になっている。

pkgs <- c("tidyverse", "lme4", "lmerTest", "mgcv", "itsadug",
          "phonTools", "randomForest", "patchwork", "here")
for (p in pkgs) {
  if (!requireNamespace(p, quietly = TRUE)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(lme4)
library(lmerTest)
library(here)

set.seed(20260101)   # 乱数を使う処理(ブートストラップ、MCMC)の再現のため(第9章参照)

theme_labphon <- function(base_size = 11) {
  theme_minimal(base_size = base_size, base_family = "notosans") +
    theme(
      panel.grid.minor = element_blank(),
      panel.grid.major = element_line(linewidth = 0.3, colour = "grey88"),
      axis.title       = element_text(size = rel(0.95)),
      strip.text       = element_text(face = "bold", size = rel(0.9)),
      legend.position  = "bottom",
      plot.title        = element_text(face = "bold", size = rel(1.05)),
      plot.caption      = element_text(size = rel(0.8), colour = "grey35")
    )
}
theme_set(theme_labphon())

# 本書のsave_fig()はPNG/PDFをassets/figures/に書き出すが、このノートブックでは
# 画面にそのまま表示すれば十分なので、print()する版に差し替える。
# (関数のシグネチャは本書と同じなので、以降のスクリプトは書き換えなしで動く。)
save_fig <- function(plot, name, width = 5.2, height = 3.6, chapter = NULL) {
  print(plot)
  invisible(NULL)
}

dir.create(here::here("data", "processed"), recursive = TRUE, showWarnings = FALSE)
# 一部のスクリプトはsave_fig()を経由せず直接ggsave()/png()でassets/figures/に
# 書き出す(このディレクトリ自体はColab上のこのセッション内でのみ存在する
# 一時的なものであり、書籍リポジトリの実ファイルには一切影響しない)。
for (chn in sprintf("ch%02d", 2:16)) {
  dir.create(here::here("assets", "figures", chn), recursive = TRUE, showWarnings = FALSE)
}

cat("セットアップ完了。実行環境:\n")
print(sessionInfo())


## `ch04_source_filter.R`

対象章: 第4章「音響分析の基礎」

「音源フィルタ理論と声道の共鳴」節の模式図。音源スペクトル(倍音列)、
声道の伝達関数(フォルマントのピーク)、そして両者を掛け合わせた出力
スペクトルを3パネルで示す、音源フィルタ理論の教科書的な図解。
理論的な模式図であり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

f0 <- 120
freq <- seq(0, 4000, by = 5)

# 音源スペクトル: 基本周波数の整数倍にピークを持つ倍音列(振幅は緩やかに減衰)
harmonics <- seq(f0, 4000, by = f0)
source_spec <- tibble(freq = freq, amp = 0)
for (h in harmonics) {
  source_spec$amp <- source_spec$amp + dnorm(source_spec$freq, mean = h, sd = 4) * (1 / (h / f0)^0.7)
}
source_spec$amp <- source_spec$amp / max(source_spec$amp)

# 声道の伝達関数: フォルマント(F1, F2, F3)に対応する共鳴ピーク
formants <- tibble(F = c(700, 1200, 2600), bw = c(70, 90, 130))
filter_resp <- tibble(freq = freq, gain = 0)
for (i in seq_len(nrow(formants))) {
  filter_resp$gain <- filter_resp$gain +
    1 / (1 + ((filter_resp$freq - formants$F[i]) / formants$bw[i])^2)
}
filter_resp$gain <- filter_resp$gain / max(filter_resp$gain)

output_spec <- tibble(freq = freq, amp = source_spec$amp * (0.15 + 0.85 * filter_resp$gain))
output_spec$amp <- output_spec$amp / max(output_spec$amp)

mk_panel <- function(df, yvar, title, colour) {
  ggplot(df, aes(x = freq, y = .data[[yvar]])) +
    geom_area(fill = colour, alpha = 0.25) +
    geom_line(colour = colour, linewidth = 0.7) +
    labs(title = title, x = "周波数 (Hz)", y = NULL) +
    scale_y_continuous(breaks = NULL) +
    theme(plot.title = element_text(size = 10, face = "bold"))
}

p1 <- mk_panel(source_spec, "amp", "音源スペクトル(声帯振動、倍音列)", "#34518A")
p2 <- mk_panel(filter_resp, "gain", "声道の伝達関数(フォルマント)", "#A8681F")
p3 <- mk_panel(output_spec, "amp", "出力スペクトル(音源×フィルタ)", "#5C7A3D")

library(patchwork)
p <- p1 / p2 / p3
ggsave(here::here("assets", "figures", "ch04", "fig04-01-source-filter.png"),
       p, width = 6, height = 6.5, dpi = 320)
ggsave(here::here("assets", "figures", "ch04", "fig04-01-source-filter.pdf"),
       p, width = 6, height = 6.5, device = cairo_pdf)
cat("完了。図はassets/figures/ch04/に保存した。\n")


## `ch04_sampling_aliasing.R`

対象章: 第4章「音響分析の基礎」

「標本化・量子化とナイキスト周波数」節の模式図。連続的な正弦波を、
ナイキスト周波数を下回る標本化周波数でサンプリングすると、実際より
低い周波数の波形として偽って記録される(エイリアシング)ことを示す。
理論的な模式図であり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

true_freq <- 900     # 真の信号の周波数(Hz)
fs_ok     <- 8000    # 十分な標本化周波数(ナイキスト周波数4000Hz > 900Hz)
fs_bad    <- 1000    # 不十分な標本化周波数(ナイキスト周波数500Hz < 900Hz)
duration  <- 0.01     # 表示区間(秒)

t_cont <- seq(0, duration, length.out = 2000)
cont_wave <- tibble(t = t_cont, amp = sin(2 * pi * true_freq * t_cont))

sample_and_alias <- function(fs, label) {
  t_samp <- seq(0, duration, by = 1 / fs)
  tibble(t = t_samp, amp = sin(2 * pi * true_freq * t_samp), fs_label = label)
}
samp_ok  <- sample_and_alias(fs_ok, paste0("fs = ", fs_ok, "Hz(ナイキスト周波数 > 信号の周波数)"))
samp_bad <- sample_and_alias(fs_bad, paste0("fs = ", fs_bad, "Hz(ナイキスト周波数 < 信号の周波数)"))

# エイリアシングで見かけ上生じる低周波成分(標本点だけを滑らかに結んだ場合に見える周波数)
alias_freq <- abs(true_freq - fs_bad * round(true_freq / fs_bad))
t_alias <- seq(0, duration, length.out = 2000)
alias_wave <- tibble(t = t_alias, amp = sin(2 * pi * alias_freq * t_alias + pi/2.2))

p1 <- ggplot() +
  geom_line(data = cont_wave, aes(t, amp), colour = "grey60", linewidth = 0.5) +
  geom_point(data = samp_ok, aes(t, amp), colour = "#34518A", size = 2) +
  geom_line(data = samp_ok, aes(t, amp), colour = "#34518A", linewidth = 0.4, linetype = 2) +
  labs(title = unique(samp_ok$fs_label), x = NULL, y = "振幅") +
  theme(plot.title = element_text(size = 9))

p2 <- ggplot() +
  geom_line(data = cont_wave, aes(t, amp), colour = "grey60", linewidth = 0.5) +
  geom_point(data = samp_bad, aes(t, amp), colour = "#A8681F", size = 2.5) +
  geom_line(data = alias_wave, aes(t, amp), colour = "#A8681F", linewidth = 0.7, linetype = 2) +
  labs(title = paste0(unique(samp_bad$fs_label), " → 見かけ上", round(alias_freq), "Hzとして記録される"),
       x = "時間 (s)", y = "振幅") +
  theme(plot.title = element_text(size = 9))

library(patchwork)
p <- p1 / p2 +
  plot_annotation(caption = paste0("真の信号: ", true_freq, "Hzの正弦波(灰色)。理論的な模式図。scripts/R/ch04_sampling_aliasing.R より生成。"))
ggsave(here::here("assets", "figures", "ch04", "fig04-02-sampling-aliasing.png"),
       p, width = 6.5, height = 5, dpi = 320)
ggsave(here::here("assets", "figures", "ch04", "fig04-02-sampling-aliasing.pdf"),
       p, width = 6.5, height = 5, device = cairo_pdf)
cat("エイリアシングで見かけ上生じる周波数:", round(alias_freq), "Hz\n")
cat("完了。図はassets/figures/ch04/に保存した。\n")


## `ch04_lpc_order.R`

対象章: 第4章「音響分析の基礎」

「フォルマント推定（LPC）の原理と誤りやすさ」節の作例。既知のフォルマント
(F1=750Hz, F2=1200Hz付近、母音/a/を想定)を持つ合成母音を作成し、
LPC次数を3段階(低い・適切・高い)に変えて、実際にLPC分析を行い、
次数がフォルマント候補の推定にどう影響するかを確認する。
phonToolsパッケージのvowelsynth()による合成音声であり、実録音ではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

library(phonTools)

fs_hz <- 11000
true_f1 <- 750
true_f2 <- 1200

snd <- vowelsynth(ffs = c(true_f1, true_f2, 2800, 3400), fbw = 0.08,
                   dur = 300, f0 = c(120, 110), fs = fs_hz, noise2 = 0.01)

# LPC係数から極(フォルマント候補)の周波数と帯域幅を求める
lpc_to_formants <- function(a, fs) {
  rts <- polyroot(a)
  rts <- rts[Im(rts) > 0]
  ang <- atan2(Im(rts), Re(rts))
  freqs <- ang * fs / (2 * pi)
  bw <- -fs / pi * log(abs(rts))
  keep <- bw < 400 & freqs > 50   # 帯域幅が狭く、直流成分でない極だけを候補とする
  tibble(freq = freqs[keep], bw = bw[keep]) |> arrange(freq)
}

orders <- c(6, 13, 26)
results <- map(orders, function(ord) {
  a <- lpc(snd, order = ord, fs = fs_hz, show = FALSE)
  lpc_to_formants(a, fs_hz)
})
names(results) <- paste0("次数", orders)

for (nm in names(results)) {
  cat(nm, ": ", paste(round(results[[nm]]$freq), collapse = ", "), " Hz\n", sep = "")
}

cat("\n真のフォルマント(合成に使用): F1 =", true_f1, "Hz, F2 =", true_f2, "Hz\n")

# --- スペクトル包絡の比較図 ------------------------------------------------
freq_axis <- seq(50, 5000, by = 10)
lpc_envelope <- function(a, fs, freqs) {
  z <- exp(-2i * pi * freqs / fs)
  h <- 1 / sapply(z, function(zz) sum(a * zz^(0:(length(a) - 1))))
  20 * log10(abs(h))
}

spec <- spectralslice(snd, fs = fs_hz, show = FALSE)
spec_df <- tibble(freq = spec[, 1], db = spec[, 2]) |> filter(freq <= 5000)

env_df <- map_dfr(orders, function(ord) {
  a <- lpc(snd, order = ord, fs = fs_hz, show = FALSE)
  env <- lpc_envelope(a, fs_hz, freq_axis)
  tibble(order = paste0("次数", ord), freq = freq_axis, db = env - max(env) + max(spec_df$db) - 5)
})

p <- ggplot() +
  geom_line(data = spec_df, aes(freq, db), colour = "grey70", linewidth = 0.3) +
  geom_line(data = env_df, aes(freq, db, colour = order, linetype = order), linewidth = 0.9) +
  geom_vline(xintercept = c(true_f1, true_f2), linetype = 2, colour = "grey40") +
  scale_colour_manual(values = c("次数6" = "#A8681F", "次数13" = "#34518A", "次数26" = "#5C7A3D")) +
  scale_linetype_manual(values = c("次数6" = "solid", "次数13" = "22", "次数26" = "42")) +
  labs(x = "周波数 (Hz)", y = "振幅 (dB)", colour = "LPC次数", linetype = "LPC次数",
       caption = "灰色は合成音声のスペクトル、破線は真のF1・F2。phonTools::vowelsynth()による合成音声。scripts/R/ch04_lpc_order.R より生成。")
save_fig(p, "fig04-04-lpc-order-comparison", chapter = "ch04", width = 7, height = 4)
cat("完了。図はassets/figures/ch04/に保存した。\n")


## `ch04_vot_waveform.R`

対象章: 第4章「音響分析の基礎」

「持続時間・強度・スペクトルモーメント・VOT」節の模式図。VOT
(Voice Onset Time)の定義を、閉鎖開放(バースト)から声帯振動開始
までの波形上で図示する。理論的な模式図であり、実測データではない。

In [ ]:
# 本書のscripts/R/chNN_*.Rは、それぞれ独立したRプロセスとして(00_setup.Rの直後に)実行され、本文の数値はその結果である。このノートブックのように複数のスクリプトを1つのセッションで連続実行すると、乱数列が前のセルの消費分だけ進んでしまい、自前のset.seed()を持たないスクリプトは本文と異なる値を出しうる。本文と同じ数値を再現するため、各セルの先頭で00_setup.Rと同じ乱数の種に戻す(自前のseedを持つスクリプトでは、直後にその値で上書きされるため無害)。
set.seed(20260101)

fs <- 16000
t <- seq(0, 0.25, by = 1 / fs)

burst_time  <- 0.05   # 閉鎖解放(バースト)の時刻
voice_onset <- 0.09   # 声帯振動開始の時刻(正のVOT、帯気を伴う)

wave <- numeric(length(t))
# 閉鎖区間(バースト以前): ほぼ無音
wave[t < burst_time] <- rnorm(sum(t < burst_time), 0, 0.01)
# バースト: 短い広帯域ノイズのパルス
burst_idx <- which(t >= burst_time & t < burst_time + 0.003)
wave[burst_idx] <- rnorm(length(burst_idx), 0, 0.8) * exp(-((t[burst_idx] - burst_time) / 0.001)^2)
# 帯気区間(バースト後、声帯振動開始前): 減衰する乱流性ノイズ
asp_idx <- which(t >= burst_time + 0.003 & t < voice_onset)
wave[asp_idx] <- rnorm(length(asp_idx), 0, 0.15) * exp(-(t[asp_idx] - burst_time) / 0.02)
# 声帯振動開始後: 周期的な母音波形
voi_idx <- which(t >= voice_onset)
wave[voi_idx] <- 0.6 * sin(2 * pi * 120 * (t[voi_idx] - voice_onset)) *
  (1 - exp(-(t[voi_idx] - voice_onset) / 0.01)) +
  rnorm(length(voi_idx), 0, 0.02)

d <- tibble(t = t * 1000, amp = wave)   # ミリ秒に変換

p <- ggplot(d, aes(t, amp)) +
  geom_line(colour = "#34518A", linewidth = 0.3) +
  geom_vline(xintercept = burst_time * 1000, linetype = 2, colour = "#A8681F") +
  geom_vline(xintercept = voice_onset * 1000, linetype = 2, colour = "#5C7A3D") +
  annotate("text", x = burst_time * 1000, y = 0.95, label = "閉鎖解放(バースト)", size = 3, hjust = -0.05, colour = "#A8681F") +
  annotate("text", x = voice_onset * 1000, y = 0.75, label = "声帯振動開始", size = 3, hjust = -0.05, colour = "#5C7A3D") +
  annotate("segment", x = burst_time * 1000, xend = voice_onset * 1000, y = -0.9, yend = -0.9,
           arrow = arrow(ends = "both", length = unit(0.15, "cm")), colour = "grey30") +
  annotate("text", x = mean(c(burst_time, voice_onset)) * 1000, y = -1.05,
           label = paste0("VOT = ", round((voice_onset - burst_time) * 1000), "ms"), size = 3.2, colour = "grey20") +
  scale_y_continuous(limits = c(-1.15, 1.05)) +
  labs(x = "時間 (ms)", y = "振幅",
       caption = "模式波形(実測データではない)。scripts/R/ch04_vot_waveform.R より生成。")

save_fig(p, "fig04-05-vot-waveform", chapter = "ch04", width = 7, height = 3.6)
cat("VOT =", round((voice_onset - burst_time) * 1000), "ms\n")
cat("完了。図はassets/figures/ch04/に保存した。\n")
